[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S06/S06_01_pandas_eda.ipynb)

# S06 · 01 — Conocer los datos: pandas y exploración (EDA)

**Tiempo estimado:** 60 min · **Sprint 6** · Módulo M2 (UD 2.1–2.3)

**Contexto TurisData.** El cliente nos ha pasado su histórico de reservas y quiere "meter IA".
Antes de entrenar nada hay que **mirar los datos con calma**: qué hay, qué falta, qué
distribución tienen y qué relaciones se intuyen. Eso es un *análisis exploratorio* (EDA).

**Al terminar sabrás:**
1. Cargar e inspeccionar una tabla con pandas (`read_csv`, `info`, `describe`).
2. Filtrar, agrupar y unir tablas.
3. Elegir el gráfico adecuado según el tipo de dato.
4. Sacar un primer *resumen de calidad* y tres hallazgos para el cliente.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 1. Qué hace un Data Scientist y qué tipos de dato manejamos

Un Data Scientist convierte preguntas de negocio en datos, modelos y recomendaciones.
Los datos pueden ser **estructurados** (tablas: lo de hoy), **no estructurados**
(texto, imágenes, audio) o **series temporales** (valores ordenados en el tiempo).
Dentro de una tabla, cada columna es de un tipo que condiciona qué podemos hacer con ella:

| Tipo | Ejemplo en TurisData | Gráfico habitual |
|---|---|---|
| Numérica continua | `precio_noche` | histograma, caja |
| Numérica discreta | `noches`, `adultos` | barras, histograma |
| Categórica | `canal`, `tipo_habitacion` | barras |
| Fecha | `fecha_llegada` | línea temporal |
| Objetivo (0/1) | `cancelada` | tasa por grupo |

## 2. Cargar e inspeccionar

**Idea.** Lo primero siempre es: ¿cuántas filas y columnas? ¿de qué tipo es cada una?
¿qué pinta tienen los primeros registros? Usamos `pd.read_csv(..., parse_dates=[...])`
para que las fechas se lean como fechas y no como texto.

In [ ]:
ruta = dato("reservas_turisdata.csv")
reservas = pd.read_csv(ruta, parse_dates=["fecha_reserva", "fecha_llegada"])
print(reservas.shape)
reservas.head()

In [ ]:
reservas.info()

In [ ]:
# Resumen estadístico de las columnas numéricas
reservas.describe().T.round(2)

**Ejercicio 1.** Carga `reservas_turisdata.csv` en un DataFrame llamado `df`
(con las dos fechas parseadas) y guarda en `n_filas` y `n_columnas` sus dimensiones.

In [ ]:
# TODO: usa pd.read_csv(ruta, parse_dates=[...]) y df.shape
...
assert n_filas == 6000, "Revisa la carga: deberían ser 6000 reservas"
assert n_columnas == 18, "Deberían ser 18 columnas"
assert str(df["fecha_llegada"].dtype).startswith("datetime"), "Parsea las fechas con parse_dates=[...]"

## 3. Filtrar y agrupar

**Idea.** Filtramos con condiciones booleanas (`df[df["canal"] == "directo"]`) y
resumimos por grupos con `groupby`. Como `cancelada` vale 0/1, **su media es la tasa de
cancelación**: un truco muy útil.

In [ ]:
# Ejemplo: tasa de cancelación por tipo de tarifa
df.groupby("tarifa")["cancelada"].mean().round(3)

In [ ]:
# Ejemplo: varias métricas a la vez
df.groupby("canal").agg(reservas=("id_reserva", "count"),
                        precio_medio=("precio_noche", "mean"),
                        tasa_cancelacion=("cancelada", "mean")).round(2)

**Ejercicio 2.** Calcula `tasa_por_deposito`: la tasa de cancelación (media de `cancelada`)
para cada valor de `deposito`, ordenada de mayor a menor.

In [ ]:
# TODO: groupby("deposito")["cancelada"].mean() y sort_values(ascending=False)
...
assert tasa_por_deposito.index[-1] == "total", "Con depósito total debería cancelar menos gente. Revisa el orden"
assert tasa_por_deposito.iloc[0] > tasa_por_deposito.iloc[-1], "Ordena de mayor a menor"

## 4. Unir tablas (`merge`)

**Idea.** A menudo la información está repartida en varias tablas. `merge` las une por una
columna común, como un `JOIN` de SQL. Aquí añadimos a cada reserva la **temporada** según
el mes de llegada, usando una pequeña tabla de equivalencias.

In [ ]:
df["mes_llegada"] = df["fecha_llegada"].dt.month
temporadas = pd.DataFrame({
    "mes_llegada": range(1, 13),
    "temporada": ["alta", "alta", "alta", "baja", "baja", "baja",
                  "alta", "alta", "baja", "baja", "baja", "alta"],
})
temporadas.head(4)

**Ejercicio 3.** Une `df` con `temporadas` por `mes_llegada` (unión `left`, para no perder
reservas) y guarda el resultado en `df_temp`. Después calcula `tasa_por_temporada`.

In [ ]:
# TODO: df.merge(temporadas, on="mes_llegada", how="left")
...
assert len(df_temp) == len(df), "Un merge 'left' con clave única no debe cambiar el número de filas"
assert "temporada" in df_temp.columns, "Falta la columna temporada"
assert set(tasa_por_temporada.index) == {"alta", "baja"}

## 5. Visualización exploratoria: el gráfico según el dato

**Idea.** Una regla práctica:
- **Una numérica** → histograma. ¿Simétrica? ¿Con cola? ¿Con valores raros?
- **Una categórica** → barras de recuento.
- **Numérica por categoría** → diagrama de caja (*boxplot*).
- **Dos numéricas** → dispersión.
- **Fecha** → línea en el tiempo.
- **Muchas numéricas a la vez** → mapa de calor de correlaciones.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df["precio_noche"], bins=30, ax=ejes[0], color="#2a6f97")
ejes[0].set(title="Distribución del precio por noche", xlabel="Precio por noche (€)", ylabel="Reservas")
sns.countplot(data=df, x="canal", ax=ejes[1], color="#2a6f97")
ejes[1].set(title="Reservas por canal", xlabel="Canal", ylabel="Reservas")
ejes[1].tick_params(axis="x", rotation=25)
sns.boxplot(data=df, x="tipo_habitacion", y="precio_noche", ax=ejes[2], color="#8ecae6")
ejes[2].set(title="Precio por tipo de habitación", xlabel="Tipo de habitación", ylabel="Precio por noche (€)")
plt.tight_layout()
plt.show()

In [ ]:
# Serie temporal: reservas por mes de llegada
por_mes = df.set_index("fecha_llegada").resample("MS")["id_reserva"].count()
plt.figure(figsize=(9, 3.5))
por_mes.plot(marker="o", color="#2a6f97")
plt.title("Reservas por mes de llegada")
plt.xlabel("Mes")
plt.ylabel("Reservas")
plt.show()

**Ejercicio 4.** Calcula `mediana_por_tipo`: la mediana de `precio_noche` por `tipo_habitacion`.
Luego dibuja un gráfico de barras con ella (título y ejes en español) y responde en la
variable `tipo_mas_caro` (texto) cuál es el tipo con mediana más alta.

In [ ]:
# TODO: groupby + median; después .plot(kind="bar"); idxmax() te da el más caro
...
assert tipo_mas_caro == "suite", "Revisa: el tipo con mayor mediana de precio debería ser la suite"
assert len(mediana_por_tipo) == 4, "Debe haber 4 tipos de habitación"

## 6. Relaciones entre variables numéricas

La **correlación** (entre −1 y 1) mide cuánto se mueven juntas dos variables. Ojo:
correlación **no** es causalidad, y solo detecta relaciones "en línea recta".

In [ ]:
numericas = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
             "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "cancelada"]
corr = df[numericas].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, cbar_kws={"label": "Correlación"})
plt.title("Correlaciones entre variables numéricas")
plt.show()

**Ejercicio 5.** Usando la matriz `corr`, guarda en `var_mas_ligada` el nombre de la variable
(distinta de `cancelada`) con **mayor correlación absoluta** con `cancelada`.

In [ ]:
# TODO: corr["cancelada"].drop("cancelada").abs().idxmax()
...
assert isinstance(var_mas_ligada, str) and var_mas_ligada != "cancelada", "Debe ser el nombre de otra variable"
assert var_mas_ligada in numericas, "Debe ser una de las variables de la matriz"

## 7. Calidad del dato: primer chequeo

Antes de modelar hay que saber **cuánto podemos fiarnos** de la tabla. Un buen resumen
de calidad recoge, por columna: tipo, nº de nulos, % de nulos y nº de valores distintos.

In [ ]:
def resumen_calidad(tabla):
    """Devuelve una tabla con el estado de calidad de cada columna."""
    return pd.DataFrame({
        "tipo": tabla.dtypes.astype(str),
        "nulos": tabla.isna().sum(),
        "pct_nulos": (tabla.isna().mean() * 100).round(2),
        "distintos": tabla.nunique(),
    })

resumen_calidad(df)

**Ejercicio 6.** Calcula `filas_duplicadas` (número de filas repetidas) y `pct_nulos_pais`
(porcentaje de nulos de `pais_origen`, redondeado a 1 decimal).

In [ ]:
# TODO: df.duplicated().sum() y df["pais_origen"].isna().mean() * 100
...
assert filas_duplicadas == 0, "Esta tabla limpia no debería tener duplicados"
assert 1.0 <= pct_nulos_pais <= 3.0, "El porcentaje de nulos de pais_origen debería rondar el 2 %"

## 8. Mini-reto: tres hallazgos para el cliente

Escribe en la lista `hallazgos` **tres frases** (texto) que un gerente de hotel entendería,
apoyadas en lo que has visto (por ejemplo, sobre canales, depósitos, precios o temporada).
Cada frase debe incluir al menos **una cifra** que hayas calculado.

In [ ]:
# TODO: escribe frases del estilo "Con depósito total cancela el X % frente al Y % sin depósito"
...
assert len(hallazgos) >= 3, "Escribe al menos tres hallazgos"
assert all(isinstance(h, str) and any(c.isdigit() for c in h) for h in hallazgos), "Cada hallazgo debe ser texto e incluir una cifra"

## 9. Para reflexionar

1. ¿Qué gráfico usarías para comparar el precio entre canales? ¿Y para ver si `antelacion_dias` se relaciona con cancelar?
2. Una correlación de 0,05 entre dos variables, ¿significa que no hay relación?
3. ¿Por qué usar la media de `cancelada` como tasa es cómodo? ¿Qué pasaría si `cancelada` estuviera codificada como "sí"/"no"?

## 10. Resumen: qué has aprendido
- Inspeccionar: `shape`, `info`, `describe`, `head`.
- Filtrar, agrupar (`groupby`) y unir (`merge`).
- Elegir el gráfico según el tipo de dato.
- Un resumen de calidad por columna es el primer paso antes de limpiar (siguiente práctica).